# Days 6-7: Baselines, metrics and matrix factorization
How do you score a top-10 list? Build recall@10 and NDCG@10, test them on a hand-worked example, score popularity baselines, then train ALS (matrix factorization).

**Checks:** instead of `assert`, each check prints what I got next to what I expected. If the two numbers differ, stop and find out why.

## Names used in this notebook
| Name | What it is |
|---|---|
| `train`, `val` | purchase tables (weeks 0-9 and week 10) |
| `articles` | raw articles file (for product type names) |
| `val_key` | validation answer key: one row per customer, the articles they bought, `is_cold` |
| `warm`, `cold` | the warm / cold rows of `val_key` |
| `recall_at_k`, `ndcg_at_k` | my two metric functions |
| `global_top10`, `last_week_top10` | the two popularity lists |
| `results_rows` | every model's score, collected for the results table |
| `customer_vocab`, `article_vocab` | Day 5 mappings: ID -> row / column number |
| `user_items` | sparse customers x articles matrix (times bought) |
| `model` | the trained ALS model |
| `reverse` | column number -> article ID |

In [1]:
# Setup. The OpenBLAS line must come BEFORE pandas/numpy are imported (restart the kernel after changing it):
# implicit uses its own threads, and OpenBLAS threads on top of that slow ALS down.
import os
os.environ['OPENBLAS_NUM_THREADS'] = '1'

import math
import time
import json
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import implicit
from implicit.als import AlternatingLeastSquares

C:\Users\srisu\OneDrive\Desktop\MachineLearning\Production Grade Projects\Recommendation System\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Day 6: Metrics and popularity baselines
## Step 1: Load the splits and rebuild the validation answer key

In [2]:
train = pd.read_parquet('../data/processed/train.parquet')
val = pd.read_parquet('../data/processed/val.parquet')
articles = pd.read_parquet('../data/RAW/articles.parquet')

print('Train rows:', len(train), '| expected 2,638,103')
print('Val rows:  ', len(val), '| expected 230,980')

Train rows: 2638103 | expected 2,638,103
Val rows:   230980 | expected 230,980


In [3]:
# Answer key (same as Day 3 Step 5): one row per validation customer, the articles they bought
val_key = val.groupby('customer_join_idx')['article_id'].unique().reset_index()

# is_cold (same as Day 4 Step 11): cold = bought nothing in train
train_customers = train['customer_join_idx'].unique()
val_key['is_cold'] = ~val_key['customer_join_idx'].isin(train_customers)

print('Validation customers:', len(val_key), '| expected 72,019')
print('Cold customers:      ', val_key['is_cold'].sum(), '| expected 23,512')
val_key.head(3)

Validation customers: 72019 | expected 72,019
Cold customers:       23512 | expected 23,512


,customer_join_idx,article_id,is_cold
0,2,[794321007],True
1,6,"[719530003, 448509014]",True
2,38,[734592001],False


## Step 2: Recall@10 on paper
**Example:** the customer bought **3** articles. My top-10 list contains **2** of them, at positions **1** and **7**.

recall@10 = hits in the top 10 / articles bought = 2 / 3 = **0.6667**

Recall ignores *where* the hits are: a hit at position 1 and one at position 10 count the same.

## Step 3: NDCG@10 on paper
**Gain of a hit at position p = 1 / log2(p + 1)**

| Position | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| Gain | 1.0000 | 0.6309 | 0.5000 | 0.4307 | 0.3869 | 0.3562 | 0.3333 | 0.3155 | 0.3010 | 0.2891 |

**DCG** (my actual hits at positions 1 and 7):
DCG = 1/log2(2) + 1/log2(8) = 1/1 + 1/3 = 1.0000 + 0.3333 = **1.3333**

**Ideal DCG** (best possible: all 3 bought articles at positions 1, 2, 3):
IDCG = 1/log2(2) + 1/log2(3) + 1/log2(4) = 1.0000 + 0.6309 + 0.5000 = **2.1309**

**NDCG** = DCG / IDCG = 1.3333 / 2.1309 = **0.6257**

### What if the hit at position 7 moved to position 2?
- **Recall:** still 2 hits out of 3 = **0.6667, no change**. Recall only counts hits.
- **NDCG:** DCG = 1 + 0.6309 = 1.6309, so NDCG = 1.6309 / 2.1309 = **0.7654, goes up**. NDCG rewards putting hits near the top.

That's the whole difference: **recall asks "did we find it?", NDCG asks "did we find it near the top?"**

## Step 4: `recall_at_k`

In [4]:
def recall_at_k(recommended, bought, k):
    """Share of the bought articles that appear in the first k recommendations.
    We only score customers who bought something, so bought is never empty."""
    hits = 0
    for item in recommended[:k]:      # only the first k count
        if item in bought:
            hits = hits + 1
    return hits / len(bought)

In [5]:
# Check on the paper example: bought 3, hits at positions 1 and 7
recommended = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]   # made-up article numbers
print('Recall:', round(recall_at_k(recommended, {1, 7, 99}, 10), 4), '| expected 0.6667')

Recall: 0.6667 | expected 0.6667


## Step 5: `ndcg_at_k`

In [6]:
def ndcg_at_k(recommended, bought, k):
    """NDCG@k: hits near the top count more. 1.0 = perfect list, 0.0 = no hits."""
    # Part 1: DCG = add up the gain of every hit in the first k
    dcg = 0
    position = 0
    for item in recommended[:k]:
        position = position + 1          # 1st item = position 1 (Python's index 0)
        if item in bought:
            dcg = dcg + 1 / math.log2(position + 1)

    # Part 2: ideal DCG = the best possible score (all bought articles at the top)
    ideal = 0
    perfect_hits = min(len(bought), k)   # can't place more than k items in a top-k list
    for position in range(1, perfect_hits + 1):
        ideal = ideal + 1 / math.log2(position + 1)

    # Part 3: my score as a fraction of the best possible
    return dcg / ideal

In [7]:
print('Paper example:', round(ndcg_at_k(recommended, {1, 7, 99}, 10), 4), '| expected 0.6257')
print('Perfect list: ', ndcg_at_k([1, 2, 3, 4, 5], {1, 2, 3}, 10), '| expected 1.0')
print('No hits:      ', ndcg_at_k(recommended, {50, 60}, 10), '| expected 0.0')

# The second hit moves from position 7 to position 2: recall stays the same, NDCG goes up
print('Recall:', round(recall_at_k(recommended, {1, 7, 99}, 10), 4), '->', round(recall_at_k(recommended, {1, 2, 99}, 10), 4))
print('NDCG:  ', round(ndcg_at_k(recommended, {1, 7, 99}, 10), 4), '->', round(ndcg_at_k(recommended, {1, 2, 99}, 10), 4))

Paper example: 0.6257 | expected 0.6257
Perfect list:  1.0 | expected 1.0
No hits:       0.0 | expected 0.0
Recall: 0.6667 -> 0.6667
NDCG:   0.6257 -> 0.7654


In [8]:
# Two small helpers used for every model from here on
def average(scores):
    return sum(scores) / len(scores)

results_rows = []    # every model's scores end up here, then become the results table

def add_result(model_name, group_name, recall_list, ndcg_list):
    """Average a group's scores and add one row to the results table."""
    results_rows.append({
        'Model': model_name,
        'Customers': group_name + ' (' + format(len(recall_list), ',') + ')',
        'Recall@10': round(average(recall_list), 4),
        'NDCG@10': round(average(ndcg_list), 4),
    })

## Step 6: Global top-10
**Guess (before running):** *not written down before running (the result is in the cells below)*

In [9]:
# Purchases per article over ALL train weeks (train only, no leakage), most bought first
counts = train['article_id'].value_counts()

# The 10 most bought, and their IDs as a plain list (what everyone gets recommended)
global_top10_counts = counts.head(10)
global_top10 = global_top10_counts.index.to_list()
print('Articles in the list:', len(global_top10), '| expected 10')

Articles in the list: 10 | expected 10


In [10]:
# What are they? Add the product type from the articles table
top10_table = global_top10_counts.reset_index()      # article_id becomes a normal column
top10_table = top10_table.merge(articles[['article_id', 'product_type_name']], on='article_id')
top10_table

,article_id,count,product_type_name
0,706016001,3853,Trousers
1,751471001,3778,Trousers
2,372860002,3464,Socks
3,730683050,3112,Unknown
4,610776002,2998,T-shirt
5,372860001,2750,Socks
6,760084003,2614,Trousers
7,759871002,2524,Vest top
8,827968001,2500,T-shirt
9,717490064,2465,T-shirt


## Step 7: Last-week top-10
**Guess (before running):** *not written down before running (the result is in the cells below)*

In [11]:
# Only week 9, the last train week
last_week = train[train['week'] == 9]
print('Week 9 purchases:', len(last_week), '| expected 240,845')

# head(10) BEFORE to_list, otherwise the list holds every article
last_week_top10_counts = last_week['article_id'].value_counts().head(10)
last_week_top10 = last_week_top10_counts.index.to_list()
print('Articles in the list:', len(last_week_top10), '| expected 10')

Week 9 purchases: 240845 | expected 240,845
Articles in the list: 10 | expected 10


In [12]:
# How many are in both lists?
overlap = 0
for item in last_week_top10:
    if item in global_top10:
        overlap = overlap + 1
print('Overlap with the global top 10:', overlap, 'of 10')

last_week_table = last_week_top10_counts.reset_index()
last_week_table = last_week_table.merge(articles[['article_id', 'product_type_name']], on='article_id')
last_week_table['in_global_top10'] = last_week_table['article_id'].isin(global_top10)
last_week_table

Overlap with the global top 10: 2 of 10


,article_id,count,product_type_name,in_global_top10
0,915526001,714,Sweater,False
1,751471043,662,Trousers,False
2,751471001,632,Trousers,True
3,706016001,568,Trousers,True
4,915529003,552,Sweater,False
5,918292001,519,Leggings/Tights,False
6,919365008,515,Trousers,False
7,863595006,511,Cardigan,False
8,896152002,490,T-shirt,False
9,762846031,430,Shirt,False


## Step 8: Score both lists on validation
**Guess (before running):** *not written down before running (the result is in the cells below)*

Each customer is scored separately, then averaged, so every customer counts equally.

In [13]:
# One score per customer, for each list and each metric (everyone gets the same list)
global_recall = []
global_ndcg = []
last_week_recall = []
last_week_ndcg = []

for bought in val_key['article_id']:
    bought = set(bought)                     # one customer's articles as a set (fast "in" checks)
    global_recall.append(recall_at_k(global_top10, bought, 10))
    global_ndcg.append(ndcg_at_k(global_top10, bought, 10))
    last_week_recall.append(recall_at_k(last_week_top10, bought, 10))
    last_week_ndcg.append(ndcg_at_k(last_week_top10, bought, 10))

print('Customers scored:', len(global_recall), '| expected 72,019')
print('Global top 10:    recall@10 =', round(average(global_recall), 4), '  NDCG@10 =', round(average(global_ndcg), 4))
print('Last-week top 10: recall@10 =', round(average(last_week_recall), 4), '  NDCG@10 =', round(average(last_week_ndcg), 4))

Customers scored: 72019 | expected 72,019
Global top 10:    recall@10 = 0.009   NDCG@10 = 0.0069
Last-week top 10: recall@10 = 0.0184   NDCG@10 = 0.0118


## Step 9: Warm vs cold
**Prediction (before running):** *not written down before running (the result is in the cells below)*

In [14]:
# Attach each customer's scores as columns (same order as the rows of val_key)
val_key['global_recall'] = global_recall
val_key['global_ndcg'] = global_ndcg
val_key['last_week_recall'] = last_week_recall
val_key['last_week_ndcg'] = last_week_ndcg

warm = val_key[~val_key['is_cold']]
cold = val_key[val_key['is_cold']]
print('Warm:', len(warm), '| expected 48,507')
print('Cold:', len(cold), '| expected 23,512')

Warm: 48507 | expected 48,507
Cold: 23512 | expected 23,512


In [15]:
# Add the popularity rows to the results table: all, warm, cold
add_result('Global top 10', 'All', global_recall, global_ndcg)
add_result('Global top 10', 'Warm', warm['global_recall'].to_list(), warm['global_ndcg'].to_list())
add_result('Global top 10', 'Cold', cold['global_recall'].to_list(), cold['global_ndcg'].to_list())

add_result('Last-week top 10', 'All', last_week_recall, last_week_ndcg)
add_result('Last-week top 10', 'Warm', warm['last_week_recall'].to_list(), warm['last_week_ndcg'].to_list())
add_result('Last-week top 10', 'Cold', cold['last_week_recall'].to_list(), cold['last_week_ndcg'].to_list())

pd.DataFrame(results_rows)

,Model,Customers,Recall@10,NDCG@10
0,Global top 10,"All (72,019)",0.0090,0.0069
1,Global top 10,"Warm (48,507)",0.0085,0.0063
2,Global top 10,"Cold (23,512)",0.0102,0.0081
3,Last-week top 10,"All (72,019)",0.0184,0.0118
4,Last-week top 10,"Warm (48,507)",0.0179,0.0116
5,Last-week top 10,"Cold (23,512)",0.0194,0.0124


In [16]:
# Same list for everyone, yet cold customers score higher. Is it basket size?
print('Average articles bought, warm:', round(warm['article_id'].apply(len).mean(), 2))
print('Average articles bought, cold:', round(cold['article_id'].apply(len).mean(), 2))

Average articles bought, warm: 3.15
Average articles bought, cold: 3.2


## Results table (every later model gets added here)

| Model | Customers | Recall@10 | NDCG@10 |
|---|---|---|---|
| Global top 10 | All (72,019) | 0.0090 | 0.0069 |
| Global top 10 | Warm (48,507) | 0.0085 | 0.0063 |
| Global top 10 | Cold (23,512) | 0.0102 | 0.0081 |
| Last-week top 10 | All (72,019) | 0.0184 | 0.0118 |
| Last-week top 10 | Warm (48,507) | 0.0179 | 0.0116 |
| Last-week top 10 | Cold (23,512) | 0.0194 | 0.0124 |

**Why do cold customers score higher with the same list?** The model treats everyone identically, so the difference comes from the customers. New customers buy the bestsellers slightly more often, while customers with 10 weeks of history have more personal, niche tastes that a one-size-fits-all list misses. It isn't basket size: both groups bought about 3.2 articles. This also means popularity is a reasonable cold-start fallback, and the warm customers are where personalisation (Day 7 onwards) has to earn its keep.

## Day 6 interview question
*"Why do recommender teams report NDCG as well as recall? Give an example where the two disagree."*

> Recall asks *did we find it?*; NDCG asks *did we find it near the top?* People look at the top of a list first, so a hit at position 1 is worth more than one at position 10, and recall can't see that. Example: a customer bought 3 articles and my list hits 2 of them at positions 1 and 7. Recall is 2/3 = 0.667 and NDCG is 0.626. Move the second hit from 7 to 2 and recall stays 0.667, but NDCG rises to 0.765. Two models with the same recall can give very different experiences, and NDCG is what tells them apart.

# Day 7: Matrix factorization
Popularity gives everyone the same list. Matrix factorization is the first **personal** model: every customer and every article gets a short learned vector, and a pair's score is the dot product of their vectors. The two-tower model (Day 9) does the same, but also builds the vectors from features.

## Step 1: A score is a dot product (on paper)
Vectors with 2 numbers:
- Customer: [0.9, 0.1]
- Article A (basic trousers): [0.8, 0.2]
- Article B (party dress): [0.1, 0.9]

**Dot product = multiply matching positions, then add:**
- Customer · A = 0.9 × 0.8 + 0.1 × 0.2 = 0.72 + 0.02 = **0.74**
- Customer · B = 0.9 × 0.1 + 0.1 × 0.9 = 0.09 + 0.09 = **0.18**

**Article A wins** (0.74 > 0.18). The customer and the trousers point in the same direction.

Each position is a hidden "taste direction" the model discovers by itself; nobody tells it that position 1 means "basics".

**Why factorization fits in memory:**
- Full customer × article table: 444,431 × 38,603 = **17,156,369,893 cells** (about 17 billion)
- Factorized with 64 numbers per vector: (444,431 + 38,603) × 64 = 483,034 × 64 = **30,914,176 numbers** (about 31 million)
- About **555 times smaller**. A huge, mostly empty table squeezed into two small ones.

## Step 2: Why bought-twice counts more (on paper)
ALS treats every cell with two numbers:
- **Preference:** 1 if bought, 0 if not
- **Confidence:** how sure we are, confidence = 1 + α × (times bought), with α = 40

| Times bought | Preference | Confidence |
|---|---|---|
| 0 | 0 | 1 + 40 × 0 = **1** |
| 1 | 1 | 1 + 40 × 1 = **41** |
| 3 | 1 | 1 + 40 × 3 = **121** |

**Which mistakes does the model care about most?** Mistakes on articles the customer actually bought, especially ones bought several times: getting a bought-3-times cell wrong costs 121 times more than getting an empty cell wrong, because an empty cell is weak evidence (the customer probably never saw the article) while a repeat purchase is strong evidence.

**How ALS trains:** **A**lternating **L**east **S**quares. Hold the article vectors fixed and solve the best customer vectors exactly (like one linear regression per customer); then hold the customer vectors fixed and solve the articles. Repeat a few times. Each half-step has an exact answer, which is why it's fast.

**Note on "Unknown":** H&M has a real product type literally called **"Unknown"**. That is a genuine category value with its own vocabulary number, different from my row 0 = unknown (never seen in training). They never collide.

## Step 3: Build the sparse matrix

In [17]:
# Day 5 mappings: ID -> row / column number (0 = unknown)
FEATURES_FOLDER = '../data/processed/features'

def load_vocab(name):
    table = pd.read_parquet(FEATURES_FOLDER + '/vocab_' + name + '.parquet')
    return dict(zip(table['value'], table['index']))

customer_vocab = load_vocab('customer')
article_vocab = load_vocab('article')
print('Customers:', len(customer_vocab), '| expected 444,431')
print('Articles: ', len(article_vocab), '| expected 38,603')

Customers:

 444431 | expected 444,431
Articles:  38603 | expected 38,603


In [18]:
# Times bought per (customer, article). train has no same-day duplicates, so each row = one different day
pair_counts = train.groupby(['customer_join_idx', 'article_id']).size().reset_index(name='times_bought')

# IDs -> row and column numbers, using the mappings (raw article IDs would make ~959 million columns)
pair_counts['row'] = pair_counts['customer_join_idx'].map(customer_vocab)
pair_counts['col'] = pair_counts['article_id'].map(article_vocab)

print('Pairs:', len(pair_counts), '| expected 2,562,634')
print('Missing rows/cols:', pair_counts['row'].isna().sum(), pair_counts['col'].isna().sum(), '| expected 0 0')

Pairs: 2562634 | expected 2,562,634
Missing rows/cols: 0 0 | expected 0 0


In [19]:
# implicit 0.7.3: fit() expects customers x articles (users x items); before 0.5 it was the reverse.
# +1 on both sides: row 0 and column 0 stay empty for "unknown", matching the mappings.
user_items = csr_matrix(
    (pair_counts['times_bought'].astype('float32'), (pair_counts['row'], pair_counts['col'])),
    shape=(len(customer_vocab) + 1, len(article_vocab) + 1),
)

## Step 4: Check the matrix

In [20]:
print('Shape:       ', user_items.shape, '| expected (444432, 38604)')
print('Filled cells:', user_items.nnz, '| expected 2,562,634 (unique pairs)')
print('Sum of cells:', int(user_items.sum()), '| expected', len(train), '(every train row adds 1 to one cell)')

# One real cell, checked by hand: proves rows and columns line up with the right people and articles
customer = 21
article = 917434002
counted_in_train = len(train[(train['customer_join_idx'] == customer) & (train['article_id'] == article)])
in_matrix = user_items[customer_vocab[customer], article_vocab[article]]
print('Customer 21 x article 917434002 -> train:', counted_in_train, '| matrix:', in_matrix)

Shape:        (444432, 38604) | expected (444432, 38604)
Filled cells: 2562634 | expected 2,562,634 (unique pairs)
Sum of cells: 2638103 | expected 2638103 (every train row adds 1 to one cell)
Customer 21 x article 917434002 -> train: 2 | matrix: 2.0


## Step 5: Train ALS
| Setting | Meaning | Value | Why |
|---|---|---|---|
| `factors` | numbers per vector (taste directions) | 64 | enough to capture many tastes, small enough to train fast |
| `iterations` | back-and-forth rounds | 15 | ALS usually settles in 10-20 rounds |
| `regularization` | penalty that stops vectors fitting noise | 0.05 | stops a customer with 1 purchase getting extreme numbers |
| `alpha` | how much a purchase boosts confidence | 40 | Step 2; the library applies it itself |
| `random_state` | fixes the random start | 42 | same result every run |

**Not tuned today**, tuning is Day 10.

**How my version handles alpha:** in implicit 0.7.3, `fit()` multiplies the matrix values by `alpha` itself (`Cui = self.alpha * Cui` in `implicit/cpu/als.py`), so confidence = 1 + alpha x times bought. I pass the raw counts and set `alpha=40`, and do not multiply the matrix myself (that would apply it twice).

In [21]:
model = AlternatingLeastSquares(
    factors=64,
    iterations=15,
    regularization=0.05,
    alpha=40,            # the library applies alpha itself, so the matrix holds raw counts
    random_state=42,
    use_gpu=False,
)

start = time.time()
model.fit(user_items)
print('ALS fit time (seconds):', round(time.time() - start, 1))

print('user_factors:', model.user_factors.shape, '| expected (444432, 64)')
print('item_factors:', model.item_factors.shape, '| expected (38604, 64)')

  0%|          | 0/15 [00:00<?, ?it/s]

  7%|▋         | 1/15 [00:02<00:29,  2.13s/it]

 13%|█▎        | 2/15 [00:04<00:27,  2.15s/it]

 20%|██        | 3/15 [00:06<00:25,  2.13s/it]

 27%|██▋       | 4/15 [00:08<00:24,  2.19s/it]

 33%|███▎      | 5/15 [00:10<00:21,  2.17s/it]

... (8 outputs left out)

 93%|█████████▎| 14/15 [00:31<00:02,  2.28s/it]

100%|██████████| 15/15 [00:33<00:00,  2.27s/it]

100%|██████████| 15/15 [00:33<00:00,  2.23s/it]

ALS fit time (seconds): 33.7
user_factors: (444432, 64) | expected (444432, 64)
item_factors: (38604, 64) | expected (38604, 64)


## Step 6: Look inside the model with one customer

In [22]:
# Customer 21 (from the Step 4 cell check). The model only knows ROW numbers.
customer = 21
customer_row = customer_vocab[customer]

columns, scores = model.recommend(
    customer_row,
    user_items[customer_row],
    N=10,
    filter_already_liked_items=False,   # show everything for now
)
print('Columns:', columns)
print('Scores: ', scores.round(3))

Columns: [30156 32619 13514 25588  7521 37656 36906  3047  2185  3048]
Scores:  [0.845 0.816 0.711 0.675 0.673 0.672 0.666 0.66  0.652 0.648]


In [23]:
# Column numbers -> article IDs: position c of this array holds the article for column c (0 = unknown)
reverse = np.zeros(len(article_vocab) + 1, dtype='int64')
for article, column in article_vocab.items():
    reverse[column] = article

article_to_type = dict(zip(articles['article_id'], articles['product_type_name']))

recommended_articles = reverse[columns]
for article, score in zip(recommended_articles, scores):
    print(article, article_to_type[article], round(float(score), 3))

863937003 Vest top 0.845
875469002 Vest top 0.816
749699002 Vest top 0.711
835851001 Blouse 0.675
685814048 Hoodie 0.673
917434002 Vest top 0.672
907696004 Vest top 0.666
572797001 Vest top 0.66
547780003 Trousers 0.652
572797002 Vest top 0.648


In [24]:
# Do the scoring myself: one dot product per article, then the top 10
user_vector = model.user_factors[customer_row]           # this customer's 64 numbers
all_scores = user_vector @ model.item_factors.T          # 38,604 scores at once
my_top10 = np.argsort(all_scores)[::-1][:10]             # highest first

print('Mine:   ', my_top10.tolist())
print('Library:', columns.tolist())
print('Same 10 articles?', list(my_top10) == list(columns), '| expected True')

Mine:    [30156, 32619, 13514, 25588, 7521, 37656, 36906, 3047, 2185, 3048]
Library: [30156, 32619, 13514, 25588, 7521, 37656, 36906, 3047, 2185, 3048]
Same 10 articles? True | expected True


In [25]:
# What did customer 21 buy in train, next to what ALS recommends?
bought_by_21 = set(train[train['customer_join_idx'] == customer]['article_id'])

print('BOUGHT in train:')
for article in train[train['customer_join_idx'] == customer]['article_id']:
    print(' ', article, article_to_type[article])

print('RECOMMENDED by ALS:')
for article in recommended_articles:
    note = '(already bought)' if article in bought_by_21 else ''
    print(' ', article, article_to_type[article], note)

BOUGHT in train:
  917434002 Vest top
  685814048 Hoodie
  895418003 Vest top
  875469002 Vest top
  835851001 Blouse
  870345002 Dress
  902325002 Skirt
  926847002 T-shirt
  909924004 Pyjama bottom
  917434002 Vest top
  946748003 Sweater
  722803001 Bra
RECOMMENDED by ALS:
  863937003 Vest top 
  875469002 Vest top (already bought)
  749699002 Vest top 
  835851001 Blouse (already bought)
  685814048 Hoodie (already bought)
  917434002 Vest top (already bought)
  907696004 Vest top 
  572797001 Vest top 
  547780003 Trousers 
  572797002 Vest top 


**Does it make sense?** Customer 21's most-bought type is vest tops (4 of 12 purchases), and ALS recommends 7 vest tops, learned from purchases alone. Two odd things: 4 of the 10 are articles they already bought (filtering is off here), and 572797001 / 572797002 are the same vest top in two colours (same product_code), the Day 6 socks problem again.

## Step 7: Score ALS on validation, with a fallback
Warm customers get ALS; cold customers (row 0, never trained) get the last-week top 10. A hybrid.

**Guess (before running):** *not written down before running (the result is in the cells below)*

In [26]:
# Warm customers' row numbers (.to_numpy() because model.recommend wants a numpy array)
warm_rows = warm['customer_join_idx'].map(customer_vocab).to_numpy()


def recommend_warm(filter_flag):
    """ALS top 10 for every warm customer, as article IDs (one row per warm customer).
    48,507 x 38,604 scores at once is too much memory, so 5,000 customers at a time."""
    chunks = []
    for start in range(0, len(warm_rows), 5000):
        chunk_rows = warm_rows[start:start + 5000]
        chunk_columns, chunk_scores = model.recommend(
            chunk_rows,
            user_items[chunk_rows],
            N=10,
            filter_already_liked_items=filter_flag,
        )
        chunks.append(chunk_columns)
    all_columns = np.vstack(chunks)      # stack the chunks into one table
    return reverse[all_columns]          # column numbers -> article IDs


def score_warm(recommendations):
    """Recall@10 and NDCG@10 for each warm customer (each has their own list)."""
    recall_list = []
    ndcg_list = []
    for recs, bought in zip(recommendations, warm['article_id']):
        recall_list.append(recall_at_k(list(recs), set(bought), 10))
        ndcg_list.append(ndcg_at_k(list(recs), set(bought), 10))
    return recall_list, ndcg_list

In [27]:
# Warm: ALS with the filter ON (hide what they already bought)
recall_on, ndcg_on = score_warm(recommend_warm(True))

# Cold: everyone gets the SAME fallback list, the last-week top 10
cold_recall = cold['last_week_recall'].to_list()
cold_ndcg = cold['last_week_ndcg'].to_list()

print('Warm ALS (filter on): recall =', round(average(recall_on), 4), '  NDCG =', round(average(ndcg_on), 4))
print('Cold fallback:        recall =', round(average(cold_recall), 4), '| expected 0.0194 (same as Day 6)')
print('                      NDCG   =', round(average(cold_ndcg), 4), '| expected 0.0124 (same as Day 6)')

# "All" = join the two lists, then one average (warm has twice as many customers)
add_result('ALS (filter on) + fallback', 'All', recall_on + cold_recall, ndcg_on + cold_ndcg)
add_result('ALS (filter on) + fallback', 'Warm', recall_on, ndcg_on)
add_result('ALS (filter on) + fallback', 'Cold', cold_recall, cold_ndcg)

Warm ALS (filter on): recall = 0.0221   NDCG = 0.0154
Cold fallback:        recall = 0.0194 | expected 0.0194 (same as Day 6)
                      NDCG   = 0.0124 | expected 0.0124 (same as Day 6)


## Step 8: Filter or no filter?
- **Filter on:** all 10 slots go to articles the customer hasn't bought; rebuyers are missed.
- **Filter off:** rebuyers are caught, but past purchases may crowd out new suggestions.

**Prediction (before running):** *not written down before running (the result is in the cells below)*

In [28]:
# Warm: ALS with the filter OFF (already-bought articles allowed)
recs_off = recommend_warm(False)
recall_off, ndcg_off = score_warm(recs_off)

print('Warm, filter ON:  recall =', round(average(recall_on), 4), '  NDCG =', round(average(ndcg_on), 4))
print('Warm, filter OFF: recall =', round(average(recall_off), 4), '  NDCG =', round(average(ndcg_off), 4))

Warm, filter ON:  recall = 0.0221   NDCG = 0.0154
Warm, filter OFF: recall = 0.0391   NDCG = 0.0273


In [29]:
# With the filter off, how many of the 10 slots go to articles the customer already bought?
already_bought_slots = []
for recs, warm_row in zip(recs_off, warm_rows):
    bought_columns = user_items[warm_row].indices         # columns this customer bought in train
    bought_articles = set(reverse[bought_columns])
    count = 0
    for article in recs:
        if article in bought_articles:
            count = count + 1
    already_bought_slots.append(count)

print('Average already-bought articles in the top 10:', round(average(already_bought_slots), 2))

Average already-bought articles in the top 10: 2.07


**Decision: filter OFF (allow already-bought articles).**

| Warm customers | Recall@10 | NDCG@10 |
|---|---|---|
| Filter on | 0.0221 | 0.0154 |
| **Filter off** | **0.0391** | **0.0273** |

Filter off is about **+77% recall and +77% NDCG** for warm customers. With the filter off, ALS fills about 2 of the 10 slots with articles the customer already bought, and those slots pay off: a specific past purchase is roughly twice as likely to be bought again next week as one of ALS's new suggestions is to be right. So even though only about 4% of warm validation purchases are repeats (Day 4), giving a couple of slots to past purchases is the best use of them. This confirms the Day 4 decision not to hard-code the filter. Cold customers are unaffected (asserted). From now on, ALS uses `filter_already_liked_items=False`.

## Step 9: Update the results table

In [30]:
add_result('ALS (filter off) + fallback ★', 'All', recall_off + cold_recall, ndcg_off + cold_ndcg)
add_result('ALS (filter off) + fallback ★', 'Warm', recall_off, ndcg_off)
add_result('ALS (filter off) + fallback ★', 'Cold', cold_recall, cold_ndcg)

results = pd.DataFrame(results_rows)

# % improvement over last-week popularity for the same customer group
last_week_recall_by_group = {'All': 0.0184, 'Warm': 0.0179, 'Cold': 0.0194}
last_week_ndcg_by_group = {'All': 0.0118, 'Warm': 0.0116, 'Cold': 0.0124}

recall_change = []
ndcg_change = []
for i in range(len(results)):
    group = results['Customers'][i].split(' ')[0]           # 'All', 'Warm' or 'Cold'
    base_recall = last_week_recall_by_group[group]
    base_ndcg = last_week_ndcg_by_group[group]
    recall_change.append(round((results['Recall@10'][i] - base_recall) / base_recall * 100))
    ndcg_change.append(round((results['NDCG@10'][i] - base_ndcg) / base_ndcg * 100))

results['Recall vs last-week %'] = recall_change
results['NDCG vs last-week %'] = ndcg_change
results

,Model,Customers,Recall@10,NDCG@10,Recall vs last-week %,NDCG vs last-week %
0,Global top 10,"All (72,019)",0.0090,0.0069,-51,-42
1,Global top 10,"Warm (48,507)",0.0085,0.0063,-53,-46
2,Global top 10,"Cold (23,512)",0.0102,0.0081,-47,-35
3,Last-week top 10,"All (72,019)",0.0184,0.0118,0,0
4,Last-week top 10,"Warm (48,507)",0.0179,0.0116,0,0
5,Last-week top 10,"Cold (23,512)",0.0194,0.0124,0,0
6,ALS (filter on) + fallback,"All (72,019)",0.0212,0.0144,15,22
7,ALS (filter on) + fallback,"Warm (48,507)",0.0221,0.0154,23,33
8,ALS (filter on) + fallback,"Cold (23,512)",0.0194,0.0124,0,0
9,ALS (filter off) + fallback ★,"All (72,019)",0.0327,0.0224,78,90


## Step 10: Save the model, the simple way
Serving only needs dot products (proved in Step 6), so I save just the numbers: two `.npy` arrays plus a small settings file. Plain arrays load anywhere, with no `implicit` needed, and don't break when the library version changes (a pickle would).

In [31]:
MODEL_FOLDER = '../models/als'
os.makedirs(MODEL_FOLDER, exist_ok=True)

np.save(MODEL_FOLDER + '/user_factors.npy', model.user_factors)
np.save(MODEL_FOLDER + '/item_factors.npy', model.item_factors)

settings = {
    'factors': 64,
    'iterations': 15,
    'regularization': 0.05,
    'alpha': 40,
    'random_state': 42,
    'filter_already_liked_items': False,   # Step 8 decision
    'implicit_version': implicit.__version__,
}
with open(MODEL_FOLDER + '/settings.json', 'w') as f:
    json.dump(settings, f, indent=2)

print(sorted(os.listdir(MODEL_FOLDER)))

['item_factors.npy', 'settings.json', 'user_factors.npy']


In [32]:
# Parity check: the SAVED numbers must give the same answers as the trained model
user_factors_loaded = np.load(MODEL_FOLDER + '/user_factors.npy')
item_factors_loaded = np.load(MODEL_FOLDER + '/item_factors.npy')
print('dtype:', user_factors_loaded.dtype, item_factors_loaded.dtype, '| expected float32 float32')

# Customer 21's top 10 from the reloaded arrays, with my own dot product (Step 6)
reloaded_scores = user_factors_loaded[customer_row] @ item_factors_loaded.T
reloaded_top10 = np.argsort(reloaded_scores)[::-1][:10]
print('Same top 10 as Step 6?', list(reloaded_top10) == list(columns), '| expected True')

dtype: float32 float32 | expected float32 float32
Same top 10 as Step 6? True | expected True


## Day 7 interview question
*"What can matrix factorization not do that a two-tower model can?"*

> Matrix factorization only learns a vector for an ID it saw in training. A new customer maps to row 0, which was never trained, so ALS can't personalise for them; that's why a third of my validation customers fall back to popularity. The same goes for an article that arrives tomorrow: it has no column until I retrain. And ALS knows nothing about the customer or the article except purchases: not age, not club status, not product type or colour. A two-tower model builds each vector from features as well as the ID, so a brand-new 22-year-old customer or a new colour of an existing product still gets a sensible vector from day one, and I can compute vectors for new articles without retraining the whole model.

# Day 8 Step 1: Repurchase baseline
Filter-off ALS scored 0.0391 for warm customers. How much of that is ALS learning **taste**, and how much is just **"show people what they bought before"**? A baseline that does *only* the second thing answers it.

**The recipe, for each warm customer:**
1. Their train purchases, most recent first
2. Each article once, at most 10
3. Fewer than 10? Fill up with the last-week top 10, skipping articles already in the list

Cold customers get the last-week top 10, as before.

**Guess (before running):** *not written down before running (the result is in the cells below)*

In [33]:
def fill_up(my_list, fallback, k=10):
    """Add fallback articles (skipping ones already in my_list) until the list has k articles."""
    filled = list(my_list)            # a copy, so the original list doesn't change
    for article in fallback:
        if len(filled) == k:
            break
        if article not in filled:
            filled.append(article)
    return filled

# Test on made-up numbers: 3 past purchases, and article 2 is also in the fallback
test = fill_up([7, 2, 9], [1, 2, 3, 4, 5, 6, 8, 10, 11, 12])
print('Got:     ', test)
print('Expected: [7, 2, 9, 1, 3, 4, 5, 6, 8, 10]')
print('Length:', len(test), '| expected 10')

Got:      [7, 2, 9, 1, 3, 4, 5, 6, 8, 10]
Expected: [7, 2, 9, 1, 3, 4, 5, 6, 8, 10]
Length: 10 | expected 10


In [34]:
# 1. Most recent purchase first
history = train.sort_values('t_dat', ascending=False)

# 2. Each article once per customer. keep='first' keeps the top copy = the most recent one
history = history.drop_duplicates(subset=['customer_join_idx', 'article_id'], keep='first')

# 3. At most 10 per customer (head keeps the order from step 1)
history = history.groupby('customer_join_idx').head(10)

# One list per customer: {customer: [newest article, ..., oldest]}
past_purchases = history.groupby('customer_join_idx')['article_id'].apply(list).to_dict()

print('Customers with a history:', len(past_purchases), '| expected 444,431 (everyone in train)')
print('Customer 21:', past_purchases[21])

Customers with a history: 444431 | expected 444,431 (everyone in train)
Customer 21: [722803001, 946748003, 917434002, 909924004, 685814048, 895418003, 875469002, 835851001, 870345002, 902325002]


In [35]:
# Every warm customer: their own past articles, topped up with last week's bestsellers
repurchase_recs = []
own_articles = []                      # how many of the 10 slots are the customer's own past articles
for customer in warm['customer_join_idx']:
    mine = past_purchases[customer]
    own_articles.append(len(mine))
    repurchase_recs.append(fill_up(mine, last_week_top10))

# Trap check: every list must really have 10 articles
lengths = [len(recs) for recs in repurchase_recs]
print('Shortest list:', min(lengths), '| expected 10')
print('Longest list: ', max(lengths), '| expected 10')
print('Average own past articles per list:', round(average(own_articles), 2))

Shortest list: 10 | expected 10
Longest list:  10 | expected 10
Average own past articles per list: 6.02


In [36]:
repurchase_recall, repurchase_ndcg = score_warm(repurchase_recs)

print('Warm, repurchase:       recall =', round(average(repurchase_recall), 4), '  NDCG =', round(average(repurchase_ndcg), 4))
print('Warm, ALS (filter off): recall =', round(average(recall_off), 4), '  NDCG =', round(average(ndcg_off), 4))
print('Warm, last-week top 10: recall =', round(average(warm['last_week_recall']), 4), '  NDCG =', round(average(warm['last_week_ndcg']), 4))

Warm, repurchase:       recall = 0.0584   NDCG = 0.0457
Warm, ALS (filter off): recall = 0.0391   NDCG = 0.0273
Warm, last-week top 10: recall = 0.0179   NDCG = 0.0116


In [37]:
# Add to the results table (run this cell once, or the rows appear twice)
add_result('Repurchase + last-week fill', 'All', repurchase_recall + cold_recall, repurchase_ndcg + cold_ndcg)
add_result('Repurchase + last-week fill', 'Warm', repurchase_recall, repurchase_ndcg)
add_result('Repurchase + last-week fill', 'Cold', cold_recall, cold_ndcg)

pd.DataFrame(results_rows)

,Model,Customers,Recall@10,NDCG@10
0,Global top 10,"All (72,019)",0.0090,0.0069
1,Global top 10,"Warm (48,507)",0.0085,0.0063
2,Global top 10,"Cold (23,512)",0.0102,0.0081
3,Last-week top 10,"All (72,019)",0.0184,0.0118
4,Last-week top 10,"Warm (48,507)",0.0179,0.0116
5,Last-week top 10,"Cold (23,512)",0.0194,0.0124
6,ALS (filter on) + fallback,"All (72,019)",0.0212,0.0144
7,ALS (filter on) + fallback,"Warm (48,507)",0.0221,0.0154
8,ALS (filter on) + fallback,"Cold (23,512)",0.0194,0.0124
9,ALS (filter off) + fallback ★,"All (72,019)",0.0327,0.0224


**Result: repurchase beats ALS, clearly.**

| Warm customers | Recall@10 | NDCG@10 |
|---|---|---|
| Last-week top 10 | 0.0179 | 0.0116 |
| ALS (filter off) | 0.0391 | 0.0273 |
| **Repurchase + last-week fill** | **0.0584** | **0.0457** |

"Show them what they bought before" beats ALS by **+49% recall and +67% NDCG**, with no model at all. On average 6 of the 10 slots are the customer's own past articles, newest first, and that alone wins. So most of ALS's gain over popularity was **memory, not taste**: ALS only gave about 2 slots to past purchases, and the 8 "new" slots were worth less than more past purchases would have been.

**What this means for the two-tower model:** the bar is now 0.0584 / 0.0457 for warm customers, not popularity. If the two-tower model can't beat this, it hasn't learned taste. The likely winning shape later is a **blend**: some slots for recent past purchases, the rest for the model's new suggestions.